# M4-Guided Recovery v2
New prediction-only fallback when no primary mask is accepted. Same weights; no training.

Validation: IoU **82.40%**, Dice **88.89%**, all 40 frozen references. v2 test performance is pending. This change follows analysis of exposed v1 test errors; any rerun on those images is development testing, not independent testing.

## 1. Install in Drive
Run this cell. The original release, checkpoints and v1 test results remain untouched.

In [ ]:
from google.colab import drive
from pathlib import Path
import urllib.request, hashlib, zipfile, json, subprocess, sys

drive.mount('/content/drive')
PROJECT=Path(input('Project folder (Enter for default): ').strip() or '/content/drive/MyDrive/CameraTrap_Phase2')
RELEASE=PROJECT/'releases/Segmentation_M4_Guided_Recovery_v2'
RELEASE.parent.mkdir(parents=True,exist_ok=True)
URL='https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/releases/segmentation_m4_guided_recovery_v2/Segmentation_M4_Guided_Recovery_v2.zip'
EXPECTED_SHA256='0042209ee5f3d548175f80d000e2d13845fc616d992083d5af86b9a13ff306b5'
ARCHIVE=RELEASE.with_suffix('.zip')
sha256=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
if ARCHIVE.exists():
    assert sha256(ARCHIVE)==EXPECTED_SHA256, 'Existing archive differs'
else:
    temp=ARCHIVE.with_suffix('.download')
    urllib.request.urlretrieve(URL,temp)
    assert sha256(temp)==EXPECTED_SHA256
    temp.replace(ARCHIVE)
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
    checks=json.loads(z.read('artifact_checksums.json'))
    for name,checksum in checks.items():assert hashlib.sha256(z.read(name)).hexdigest()==checksum,name
    for entry in z.infolist():
        target=(RELEASE/entry.filename).resolve()
        assert target.is_relative_to(RELEASE.resolve())
        if target.exists() and not entry.is_dir():
            assert target.is_file() and sha256(target)==hashlib.sha256(z.read(entry.filename)).hexdigest(), f'Existing file differs: {target}'
    RELEASE.mkdir(parents=True,exist_ok=True)
    z.extractall(RELEASE)
for name,checksum in checks.items():assert sha256(RELEASE/name)==checksum,name
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.3.221'],check=True)
print('Installed:',RELEASE)
print('Saved validation results (not a new test):',json.loads((RELEASE/'validation_metrics.json').read_text()))


## 2. Fixed-policy Development Test
Run on the original 40 test images and unchanged reference masks. Enter the actual image/label folder paths or accept the defaults. Uses a new timestamped output folder. No policy or threshold selection occurs in this cell. GPU recommended; CPU also works.

In [ ]:
import pandas as pd
from datetime import datetime, timezone
import importlib.util
import numpy as np
from PIL import Image

# PROJECT is selected in the installation cell.
image_folder = Path(input('Test image folder (Enter for default): ').strip() or str(PROJECT/'segmentation/images/test'))
label_folder = Path(input('Test label folder (Enter for default): ').strip() or str(PROJECT/'segmentation/labels/test'))
images = sorted(p for p in image_folder.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png'))
assert len(images) == 40, f'Expected all 40 test images; found {len(images)}'
EXPECTED_TEST_IDS=['585a6491-23d2-11e8-a6a3-ec086b02610b', '5860ecd3-23d2-11e8-a6a3-ec086b02610b', '58678cd3-23d2-11e8-a6a3-ec086b02610b', '586fdcc3-23d2-11e8-a6a3-ec086b02610b', '586fddb6-23d2-11e8-a6a3-ec086b02610b', '5871868b-23d2-11e8-a6a3-ec086b02610b', '58732f6a-23d2-11e8-a6a3-ec086b02610b', '587d4280-23d2-11e8-a6a3-ec086b02610b', '587eea21-23d2-11e8-a6a3-ec086b02610b', '588c129e-23d2-11e8-a6a3-ec086b02610b', '58aa593f-23d2-11e8-a6a3-ec086b02610b', '58af77a1-23d2-11e8-a6a3-ec086b02610b', '58b131ce-23d2-11e8-a6a3-ec086b02610b', '58b664ca-23d2-11e8-a6a3-ec086b02610b', '58b9d6b2-23d2-11e8-a6a3-ec086b02610b', '58bb8c19-23d2-11e8-a6a3-ec086b02610b', '58bd1a8d-23d2-11e8-a6a3-ec086b02610b', '58cca4e1-23d2-11e8-a6a3-ec086b02610b', '58ddd7fe-23d2-11e8-a6a3-ec086b02610b', '58e0f7dd-23d2-11e8-a6a3-ec086b02610b', '58f73c5c-23d2-11e8-a6a3-ec086b02610b', '590d2d89-23d2-11e8-a6a3-ec086b02610b', '59104acb-23d2-11e8-a6a3-ec086b02610b', '59180b73-23d2-11e8-a6a3-ec086b02610b', '591fd0ea-23d2-11e8-a6a3-ec086b02610b', '592479bf-23d2-11e8-a6a3-ec086b02610b', '59292e51-23d2-11e8-a6a3-ec086b02610b', '592c4c68-23d2-11e8-a6a3-ec086b02610b', '592de468-23d2-11e8-a6a3-ec086b02610b', '593a5058-23d2-11e8-a6a3-ec086b02610b', '59484473-23d2-11e8-a6a3-ec086b02610b', '59641ee4-23d2-11e8-a6a3-ec086b02610b', '59862499-23d2-11e8-a6a3-ec086b02610b', '5991043a-23d2-11e8-a6a3-ec086b02610b', '59adfea1-23d2-11e8-a6a3-ec086b02610b', '59c4c0bb-23d2-11e8-a6a3-ec086b02610b', '59e2a2d1-23d2-11e8-a6a3-ec086b02610b', '59f129f9-23d2-11e8-a6a3-ec086b02610b', '5a049322-23d2-11e8-a6a3-ec086b02610b', '5a14a37d-23d2-11e8-a6a3-ec086b02610b']
assert {p.stem for p in images} == set(EXPECTED_TEST_IDS), 'Use the same original 40 test images; do not substitute other images'
labels = {p.stem: label_folder/(p.stem+'.txt') for p in images}
assert all(p.is_file() for p in labels.values()), 'Reference labels are missing'
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
policy_file = RELEASE/'refinement_policy.json'
policy_hash = sha(policy_file)
EVALUATION = RELEASE/'evaluations'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EVALUATION.mkdir(parents=True, exist_ok=False)
protocol = dict(split='test', n=40, policy_sha256=policy_hash,
                image_sha256={p.name:sha(p) for p in images},
                reference_sha256={key:sha(path) for key,path in labels.items()},
                status='Development test: original test previously inspected by team. No tuning in this cell.')
(EVALUATION/'protocol.json').write_text(json.dumps(protocol,indent=2))
subprocess.run([sys.executable,str(RELEASE/'predict_refined.py'),
                '--images',str(image_folder),'--output',str(EVALUATION/'predictions.json')],check=True)
assert sha(policy_file) == policy_hash, 'Policy changed during evaluation'
spec = importlib.util.spec_from_file_location('release_runtime',RELEASE/'predict_refined.py')
runtime = importlib.util.module_from_spec(spec);spec.loader.exec_module(runtime)
results = json.loads((EVALUATION/'predictions.json').read_text())
assert results['policy_sha256'] == policy_hash
predictions = {Path(r['image']).stem:r for r in results['results']}
assert set(predictions) == set(labels), 'Missing or duplicate image predictions'
rows=[]
for path in images:
    key=path.stem; reference=labels[key]
    assert sha(reference) == protocol['reference_sha256'][key]
    assert sha(path) == protocol['image_sha256'][path.name]
    polygons=[]
    for line in reference.read_text().splitlines():
        if not line.strip():continue
        values=[float(v) for v in line.split()]
        assert values[0] == 0 and len(values) >= 7 and len(values)%2 == 1
        assert all(np.isfinite(v) and 0 <= v <= 1 for v in values[1:])
        polygons.append(np.array(values[1:]).reshape(-1,2))
    with Image.open(path) as im:w,h=im.size
    truth=runtime.render(polygons,w,h)
    predicted=runtime.render([o['polygon_normalized'] for o in predictions[key]['objects']],w,h)
    intersection=int((truth & predicted).sum());union=int((truth | predicted).sum())
    rows.append(dict(id=key,iou=intersection/union if union else 1.,
        dice=2*intersection/int(truth.sum()+predicted.sum()) if union else 1.,
        area_mae_fraction=abs(float(truth.mean())-float(predicted.mean()))))
frame=pd.DataFrame(rows)
frame.to_csv(EVALUATION/'per_image.csv',index=False)
metrics=dict(split='test',n=len(frame),mean_iou=float(frame.iou.mean()),
             mean_dice=float(frame.dice.mean()),area_mae_fraction=float(frame.area_mae_fraction.mean()),
             policy_sha256=policy_hash,independent_test=False)
(EVALUATION/'metrics.json').write_text(json.dumps(metrics,indent=2))
print(metrics)
print('Saved:', EVALUATION)